# TP2 - Clase 3: Detector de Máximo Enfoque en Video
**Métrica:** Image Sharpness Measure for Blurred Images in Frequency Domain (De & Masilamani, 2013)

**Experimentos:**
1. Medición sobre todo el frame
2. Medición sobre ROI central (5% y 10%)
3. Matriz NxM de ROIs equiespaciadas *(opcional)*
4. Unsharp Masking *(puntos extra)*

## 0. Instalación de dependencias

In [ ]:
!pip install opencv-python numpy matplotlib scipy

## 1. Imports y configuración

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import uniform_filter1d
import os

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# =============================================
# CONFIGURACIÓN — cambiá solo esta línea
VIDEO_PATH = 'focus_video.mov'
OUT_DIR    = 'resultados_tp2'
# =============================================

os.makedirs(OUT_DIR, exist_ok=True)
print('✓ Librerías importadas correctamente')

## 2. Funciones de métricas de enfoque

In [ ]:
def to_gray(frame):
    if frame.ndim == 3:
        return cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(np.float64)
    return frame.astype(np.float64)


def fm_frequency(region):
    """
    Métrica FM — De & Masilamani (2013)
    FM = T_H / (M x N)
    T_H = píxeles del espectro de Fourier centrado > max/1000
    """
    if region.size == 0:
        return 0.0
    gray = to_gray(region)
    M, N = gray.shape
    AF = np.abs(np.fft.fftshift(np.fft.fft2(gray)))
    max_val = np.max(AF)
    if max_val == 0:
        return 0.0
    T_H = np.sum(AF > max_val / 1000.0)
    return float(T_H / (M * N))


def fm_laplacian(region):
    """
    Métrica alternativa — Modified Laplacian (LAPM)
    LAPM = mean(|∇²I|)
    """
    gray = to_gray(region)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    return float(np.mean(np.abs(lap)))


def apply_unsharp_mask(frame, kernel_size=21, sigma=3.0, strength=1.5):
    """Unsharp masking: realza bordes antes de calcular enfoque."""
    blurred = cv2.GaussianBlur(frame, (kernel_size, kernel_size), sigma)
    return cv2.addWeighted(frame, 1 + strength, blurred, -strength, 0)


def find_peak(values, smooth_window=None):
    """Detecta automáticamente el frame de máximo enfoque."""
    arr = np.array(values)
    w = smooth_window or max(3, len(arr) // 50)
    smoothed = uniform_filter1d(arr, size=w)
    peak_idx = int(np.argmax(smoothed))
    return peak_idx, smoothed


print('✓ Métricas definidas')

## 3. Funciones de regiones

In [ ]:
def get_central_roi(frame, area_fraction=0.05):
    H, W = frame.shape[:2]
    side = int(np.sqrt(H * W * area_fraction))
    cx, cy = W // 2, H // 2
    x1, y1 = max(cx - side//2, 0), max(cy - side//2, 0)
    x2, y2 = min(x1 + side, W),    min(y1 + side, H)
    return frame[y1:y2, x1:x2], (x1, y1, x2, y2)


def get_focus_grid(frame, N_rows=3, M_cols=3):
    H, W = frame.shape[:2]
    cell_h, cell_w = H // N_rows, W // M_cols
    regions = []
    for r in range(N_rows):
        for c in range(M_cols):
            y1, x1 = r * cell_h, c * cell_w
            y2, x2 = min(y1 + cell_h, H), min(x1 + cell_w, W)
            regions.append((frame[y1:y2, x1:x2], (x1, y1, x2, y2), (r, c)))
    return regions


print('✓ Funciones de regiones definidas')

## 4. Procesamiento del video (todos los experimentos)

In [ ]:
GRID_CONFIGS = [(3, 3), (7, 5), (5, 5)]

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(f'No se encontró el video: {VIDEO_PATH}')

FPS    = cap.get(cv2.CAP_PROP_FPS)
TOTAL  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Video: {VIDEO_PATH}  |  FPS: {FPS:.2f}  |  Frames: {TOTAL}')

# Almacenamiento de resultados
exp1_fm, exp1_lap           = [], []
exp2_5_fm, exp2_10_fm       = [], []
exp2_5_lap, exp2_10_lap     = [], []
exp3 = {cfg: {} for cfg in GRID_CONFIGS}

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Exp 1: frame completo
    exp1_fm.append(fm_frequency(frame))
    exp1_lap.append(fm_laplacian(frame))

    # Exp 2: ROI central
    roi5,  _ = get_central_roi(frame, 0.05)
    roi10, _ = get_central_roi(frame, 0.10)
    exp2_5_fm.append(fm_frequency(roi5));   exp2_5_lap.append(fm_laplacian(roi5))
    exp2_10_fm.append(fm_frequency(roi10)); exp2_10_lap.append(fm_laplacian(roi10))

    # Exp 3: grillas
    for cfg in GRID_CONFIGS:
        N, M = cfg
        for region, _, (r, c) in get_focus_grid(frame, N, M):
            exp3[cfg].setdefault((r, c), []).append(fm_frequency(region))

    frame_idx += 1
    if frame_idx % 30 == 0:
        print(f'  {frame_idx}/{TOTAL} frames...', end='\r')

cap.release()
N_FRAMES = frame_idx
FRAMES   = np.arange(N_FRAMES)
print(f'\n✓ Procesamiento completo — {N_FRAMES} frames')

## 5. Experimento 1 — Frame completo

In [ ]:
peak_fm,  sm_fm  = find_peak(exp1_fm)
peak_lap, sm_lap = find_peak(exp1_lap)

fig, axes = plt.subplots(2, 1, figsize=(13, 8))
fig.suptitle('Experimento 1 – Frame Completo', fontsize=14, fontweight='bold')

for ax, vals, smooth, peak, label, color in [
    (axes[0], exp1_fm,  sm_fm,  peak_fm,  'FM (De & Masilamani, 2013)', 'steelblue'),
    (axes[1], exp1_lap, sm_lap, peak_lap, 'Modified Laplacian (LAPM)',  'darkorange'),
]:
    ax.plot(FRAMES, vals,   alpha=0.35, color=color, linewidth=0.8, label='Raw')
    ax.plot(FRAMES, smooth, color=color, linewidth=2,   label='Suavizado')
    ax.axvline(peak, color='red', linestyle='--', linewidth=1.5, label=f'Máx enfoque: frame {peak}')
    ax.scatter([peak], [vals[peak]], color='red', zorder=5, s=60)
    ax.set_xlabel('Frame #'); ax.set_ylabel('Score')
    ax.set_title(label); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/exp1_frame_completo.png', dpi=150)
plt.show()
print(f'FM  → máx enfoque en frame {peak_fm}  ({peak_fm/FPS:.2f}s)')
print(f'LAP → máx enfoque en frame {peak_lap} ({peak_lap/FPS:.2f}s)')

## 6. Experimento 2 — ROI Central (5% y 10%)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Experimento 2 – ROI Central', fontsize=14, fontweight='bold')

configs = [
    (exp2_5_fm,  'FM – ROI 5%',         axes[0,0], 'steelblue'),
    (exp2_10_fm, 'FM – ROI 10%',        axes[0,1], 'navy'),
    (exp2_5_lap, 'Laplaciano – ROI 5%', axes[1,0], 'darkorange'),
    (exp2_10_lap,'Laplaciano – ROI 10%',axes[1,1], 'saddlebrown'),
]
for vals, title, ax, color in configs:
    peak, smooth = find_peak(vals)
    ax.plot(FRAMES, vals,   alpha=0.35, color=color, linewidth=0.8)
    ax.plot(FRAMES, smooth, color=color, linewidth=2, label='Suavizado')
    ax.axvline(peak, color='red', linestyle='--', linewidth=1.5, label=f'Frame {peak}')
    ax.scatter([peak], [vals[peak]], color='red', zorder=5, s=60)
    ax.set_title(title); ax.set_xlabel('Frame #'); ax.set_ylabel('Score')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/exp2_roi_central.png', dpi=150)
plt.show()

## 7. Experimento 3 — Grilla NxM (Opcional)

In [ ]:
for cfg in GRID_CONFIGS:
    N, M = cfg
    cell_data = exp3[cfg]

    cell_matrix = np.zeros((N, M, N_FRAMES))
    global_fm   = np.zeros(N_FRAMES)
    for (r, c), vals in cell_data.items():
        cell_matrix[r, c, :] = vals
        global_fm += np.array(vals)
    global_fm /= (N * M)
    best_frame, global_smooth = find_peak(global_fm.tolist())

    fig = plt.figure(figsize=(16, 4 + N * 1.5))
    fig.suptitle(f'Experimento 3 – Grilla {N}×{M}', fontsize=13, fontweight='bold')
    gs = gridspec.GridSpec(N, M + 2, figure=fig, wspace=0.4, hspace=0.5)

    # Curva global
    ax_g = fig.add_subplot(gs[:, M:M+2])
    ax_g.plot(FRAMES, global_fm,     alpha=0.4, color='teal', linewidth=0.8)
    ax_g.plot(FRAMES, global_smooth, color='teal', linewidth=2)
    ax_g.axvline(best_frame, color='red', linestyle='--', label=f'Frame {best_frame}')
    ax_g.set_title('FM Promedio'); ax_g.set_xlabel('Frame #')
    ax_g.legend(); ax_g.grid(True, alpha=0.3)

    # Curvas por celda
    for r in range(N):
        for c in range(M):
            ax = fig.add_subplot(gs[r, c])
            vals = cell_matrix[r, c, :]
            _, sm = find_peak(vals.tolist())
            ax.plot(FRAMES, vals, linewidth=0.6, color='gray')
            ax.plot(FRAMES, sm,   linewidth=1.2, color='steelblue')
            ax.axvline(best_frame, color='red', linestyle=':', linewidth=0.8)
            ax.set_title(f'({r},{c})', fontsize=7)
            ax.tick_params(labelsize=5); ax.grid(True, alpha=0.2)

    plt.savefig(f'{OUT_DIR}/exp3_grilla_{N}x{M}.png', dpi=130)
    plt.show()
    print(f'Grilla {N}×{M} → máx enfoque: frame {best_frame} ({best_frame/FPS:.2f}s)')

## 8. Frame de máximo enfoque con grilla superpuesta

In [ ]:
best_frame_global, _ = find_peak(exp1_fm)

cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, best_frame_global)
ret, best_frame_img = cap.read()
cap.release()

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle(f'Frame de máximo enfoque: #{best_frame_global} ({best_frame_global/FPS:.2f}s)',
             fontsize=13, fontweight='bold')

# Frame original
axes[0].imshow(cv2.cvtColor(best_frame_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('Frame original'); axes[0].axis('off')

# Con ROIs
vis_roi = best_frame_img.copy()
_, b5  = get_central_roi(best_frame_img, 0.05)
_, b10 = get_central_roi(best_frame_img, 0.10)
cv2.rectangle(vis_roi, b5[:2],  b5[2:],  (0,255,0), 2)
cv2.rectangle(vis_roi, b10[:2], b10[2:], (0,0,255), 2)
axes[1].imshow(cv2.cvtColor(vis_roi, cv2.COLOR_BGR2RGB))
axes[1].set_title('ROI 5% (verde) / 10% (azul)'); axes[1].axis('off')

# Con grilla 3x3
vis_33 = best_frame_img.copy()
for _, (x1,y1,x2,y2), _ in get_focus_grid(best_frame_img, 3, 3):
    cv2.rectangle(vis_33, (x1+2,y1+2), (x2-2,y2-2), (0,255,0), 2)
axes[2].imshow(cv2.cvtColor(vis_33, cv2.COLOR_BGR2RGB))
axes[2].set_title('Grilla 3×3'); axes[2].axis('off')

# Con grilla 7x5
vis_75 = best_frame_img.copy()
for _, (x1,y1,x2,y2), _ in get_focus_grid(best_frame_img, 7, 5):
    cv2.rectangle(vis_75, (x1+2,y1+2), (x2-2,y2-2), (0,255,0), 2)
axes[3].imshow(cv2.cvtColor(vis_75, cv2.COLOR_BGR2RGB))
axes[3].set_title('Grilla 7×5'); axes[3].axis('off')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/best_frame_visualizaciones.png', dpi=150)
plt.show()
print(f'✓ Guardado en {OUT_DIR}/best_frame_visualizaciones.png')

## 9. Puntos Extra — Unsharp Masking

In [ ]:
print('Procesando Unsharp Masking...')
cap = cv2.VideoCapture(VIDEO_PATH)
fm_unsharp = []
fi = 0
while True:
    ret, frame = cap.read()
    if not ret: break
    fm_unsharp.append(fm_frequency(apply_unsharp_mask(frame)))
    fi += 1
    if fi % 30 == 0: print(f'  {fi}/{N_FRAMES}...', end='\r')
cap.release()

peak_orig,   sm_orig = find_peak(exp1_fm)
peak_unsharp, sm_ush = find_peak(fm_unsharp)

fig, axes = plt.subplots(2, 1, figsize=(13, 8))
fig.suptitle('Puntos Extra – Unsharp Masking vs Original', fontsize=13, fontweight='bold')

ax = axes[0]
ax.plot(FRAMES, exp1_fm,   alpha=0.3, color='steelblue', linewidth=0.8)
ax.plot(FRAMES, sm_orig,   color='steelblue', linewidth=2, label='FM original')
ax.axvline(peak_orig,    color='steelblue', linestyle='--', linewidth=1.5, label=f'Máx original: frame {peak_orig}')
ax.plot(FRAMES, fm_unsharp, alpha=0.3, color='tomato', linewidth=0.8)
ax.plot(FRAMES, sm_ush,    color='tomato', linewidth=2, label='FM + Unsharp Mask')
ax.axvline(peak_unsharp, color='tomato', linestyle='--', linewidth=1.5, label=f'Máx unsharp: frame {peak_unsharp}')
ax.set_xlabel('Frame #'); ax.set_ylabel('FM Score')
ax.set_title('FM Original vs con Unsharp Masking'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
diff = np.array(fm_unsharp) - np.array(exp1_fm)
ax.bar(FRAMES, diff, color='purple', alpha=0.5, width=1.0)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Frame #'); ax.set_ylabel('Diferencia FM')
ax.set_title('Diferencia (Unsharp − Original): zona de enfoque expandida')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/extra_unsharp_masking.png', dpi=150)
plt.show()
print(f'\n→ Máx original:    frame {peak_orig}    ({peak_orig/FPS:.2f}s)')
print(f'→ Máx con unsharp: frame {peak_unsharp} ({peak_unsharp/FPS:.2f}s)')

## 10. Resumen de resultados

In [ ]:
print('=' * 50)
print('RESUMEN — PUNTOS DE MÁXIMO ENFOQUE')
print('=' * 50)
p1, _ = find_peak(exp1_fm);   print(f'Exp 1  FM  (frame completo):   frame {p1}  ({p1/FPS:.2f}s)')
p2, _ = find_peak(exp1_lap);  print(f'Exp 1  LAP (frame completo):   frame {p2}  ({p2/FPS:.2f}s)')
p3, _ = find_peak(exp2_5_fm); print(f'Exp 2  FM  (ROI  5%):          frame {p3}  ({p3/FPS:.2f}s)')
p4, _ = find_peak(exp2_10_fm);print(f'Exp 2  FM  (ROI 10%):          frame {p4}  ({p4/FPS:.2f}s)')
for cfg in GRID_CONFIGS:
    N, M = cfg
    gfm = np.mean([exp3[cfg][(r,c)] for r in range(N) for c in range(M)], axis=0)
    pg, _ = find_peak(gfm.tolist())
    print(f'Exp 3  FM  (grilla {N}×{M}):        frame {pg}  ({pg/FPS:.2f}s)')
print(f'Extra  FM  (unsharp):          frame {peak_unsharp}  ({peak_unsharp/FPS:.2f}s)')
print('=' * 50)
print(f'\n✓ Todos los gráficos guardados en: {OUT_DIR}/')